# HTR – Explorative Analyse

Dieses Notebook zeigt:
1. Datensatz laden und visualisieren
2. Modellarchitektur erkunden
3. CTC-Loss verstehen
4. Trainingskurven anzeigen

In [ ]:
import sys
sys.path.insert(0, '..')  # Projekt-Root zum Python-Pfad hinzufügen

import torch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA verfügbar: {torch.cuda.is_available()}')

## 1. Datensatz erkunden

In [ ]:
from src.dataset import SyntheticHTRDataset, ALPHABET, NUM_CLASSES

# Synthetischen Datensatz erstellen
dataset = SyntheticHTRDataset(size=200, img_height=32, img_width=128, split='train')
print(f'Datensatz-Größe: {len(dataset)}')
print(f'Alphabet-Größe:  {NUM_CLASSES} Klassen')
print(f'Alphabet (erste 20 Zeichen): {repr(ALPHABET[:20])}')

# Beispiel-Sample anschauen
img_tensor, label_tensor, length = dataset[0]
print(f'\nBild-Tensor Shape:  {img_tensor.shape}  (C×H×W)')
print(f'Label-Tensor:       {label_tensor}')
print(f'Label-Länge:        {length}')

In [ ]:
from src.dataset import decode_label

# 8 Beispielbilder visualisieren
fig, axes = plt.subplots(2, 4, figsize=(16, 5))
axes = axes.flatten()

for i, ax in enumerate(axes):
    img_tensor, label_tensor, _ = dataset[i]
    img_np = img_tensor.squeeze().numpy()
    img_np = (img_np * 0.5 + 0.5)  # Denormalisierung
    
    label_text = decode_label(label_tensor.tolist())
    
    ax.imshow(img_np, cmap='gray', aspect='auto')
    ax.set_title(f'Label: "{label_text}"', fontsize=8)
    ax.axis('off')

plt.suptitle('Synthetische Trainingsbilder', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Modellarchitektur

In [ ]:
from src.model import build_model

model = build_model(img_height=32, num_classes=NUM_CLASSES, lstm_hidden=256)
print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nGesamtparameter:    {total_params:,}')
print(f'Trainierbar:         {trainable:,}')

In [ ]:
# Vorwärtsdurchlauf testen
dummy_batch = torch.randn(4, 1, 32, 128)
with torch.no_grad():
    output = model(dummy_batch)

print(f'Eingabe:  {dummy_batch.shape}  (Batch × Kanäle × Höhe × Breite)')
print(f'Ausgabe:  {output.shape}       (SeqLen × Batch × NumKlassen)')
print(f'\nSequenzlänge = {output.shape[0]} (Breite nach CNN-Pooling)')
print('Diese Sequenz wird vom CTC-Decoder in Text umgewandelt.')

## 3. CTC-Loss verstehen

In [ ]:
# CTC-Loss Beispiel
ctc_loss = torch.nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)

# Zufällige Modellausgabe (Log-Softmax)
seq_len, batch, n_classes = output.shape
log_probs = output  # bereits Log-Softmax

# Beispiel-Labels: [1, 2, 3] und [4, 5] für zwei Batch-Elemente
targets      = torch.tensor([1, 2, 3, 4, 5], dtype=torch.long)
target_lens  = torch.tensor([3, 2], dtype=torch.long)
input_lens   = torch.full((2,), seq_len, dtype=torch.long)

loss = ctc_loss(log_probs[:, :2, :], targets, input_lens[:2], target_lens)
print(f'CTC-Loss (zufälliges Modell): {loss.item():.4f}')
print()
print('Erklärung: CTC summiert alle möglichen Pfade (Zeichensequenzen),')
print('die nach dem Zusammenführen von Blanks und Duplikaten zum Ziel-Label führen.')
print('Das Modell lernt, welche Zeitschritte welchen Zeichen entsprechen,')
print('ohne dass wir die genaue Ausrichtung vorgeben müssen.')

## 4. Trainingskurven (nach dem Training)

In [ ]:
import json
from pathlib import Path

history_path = Path('../outputs/logs/history.json')

if history_path.exists():
    with open(history_path) as f:
        history = json.load(f)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(history.get('train_loss', []), label='Train Loss', color='#2196F3')
    axes[0].plot(history.get('val_loss',   []), label='Val Loss',   color='#F44336', ls='--')
    axes[0].set_title('CTC Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    
    axes[1].plot(history.get('train_cer', []), label='Train CER', color='#4CAF50')
    axes[1].plot(history.get('val_cer',   []), label='Val CER',   color='#FF9800', ls='--')
    axes[1].set_title('Character Error Rate'); axes[1].legend(); axes[1].grid(alpha=0.3)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('CER')
    
    plt.suptitle('HTR Training – Verlauf', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('Noch keine Trainingsdaten. Starte zuerst: python -m training.train')

## 5. Greedy-Decoding verstehen

In [ ]:
from utils.ctc_decoder import greedy_decode
from src.dataset import IDX2CHAR

# Vorwärtsdurchlauf mit einem Synthetikbild
img_tensor, label_tensor, length = dataset[5]
with torch.no_grad():
    log_probs = model(img_tensor.unsqueeze(0))  # (SeqLen, 1, NumClasses)

decoded = greedy_decode(log_probs)
truth   = decode_label(label_tensor.tolist())

print(f'Ground Truth: "{truth}"')
print(f'Vorhersage:   "{decoded[0]}" (untrainiertes Modell → zufällig)')
print()
print('Nach dem Training wird die Vorhersage dem Ground Truth ähneln.')